In [7]:
pip install --upgrade gotabpfn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.1/116.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: gotabpfn
    Found existing installation: gotabpfn 0.1.10
    Uninstalling gotabpfn-0.1.10:
      Successfully uninstalled gotabpfn-0.1.10


In [1]:
pip show gotabpfn

Name: gotabpfn
Version: 0.1.11
Summary: GOTabPFN: From Feature Ordering to Compact Tokenization for Tabular Foundation Models on High-Dimensional Data
Home-page: https://github.com/zadid6pretam/GOTabPFN
Author: Al Zadid Sultan Bin Habib, Md Younus Ahamed, Prashnna Kumar Gyawali, Gianfranco Doretto, Donald A. Adjeroh
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: kmeans-gpu, matplotlib, numpy, optuna, pandas, scikit-learn, scipy, tabpfn, torch, tqdm
Required-by: 


# DrivFace - Regression || Single Wrapper

In [5]:
import numpy as np
import torch

from gotabpfn import run_gotabpfn_csv


# -----------------------
# User settings
# -----------------------
DATA_FILE = "drivface.csv"  # change this to your dataset file name
TARGET_COL = "angle"        # change this to your regression target column
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# -----------------------
# Fixed GO-LR hyperparameters
# -----------------------
GO_METRIC = "manhattan"
GO_NUM_CLUSTERS = 5
GO_REFINE_PASSES = 1
GO_DIRECTION_SELECT = False
GO_FEAT_SUBSAMPLE = 2000


# -----------------------
# Fixed NSC-pSP hyperparameters
# -----------------------
NSC_SEGMENTATION = "largest_jump"
NSC_M_RULE = "idf"
NSC_TAU = 0.99
NSC_GAMMA = 2.654390393837633
NSC_BETA = 0.043192175152615336
NSC_MMIN = 16
NSC_MMAX = 256
NSC_LMIN = 12
ASSUME_STANDARDIZED = True

TABPFN_SEED = 3


# -----------------------
# Run GOTabPFN
# -----------------------
results = run_gotabpfn_csv(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    task_type="regression",
    non_numeric="drop",

    # 5x5 repeated cross-validation
    cv="5x5",
    seed=SEED,

    # GO-LR settings
    go_metric=GO_METRIC,
    go_num_clusters=GO_NUM_CLUSTERS,
    go_refine=True,
    go_direction_select=GO_DIRECTION_SELECT,
    go_refine_passes=GO_REFINE_PASSES,
    go_feat_subsample=GO_FEAT_SUBSAMPLE,

    # Try GPU KMeans first, then fall back to CPU KMeans if needed
    go_use_cpu_kmeans=False,
    go_fallback_cpu_kmeans=True,

    # NSC-pSP settings
    nsc_segmentation=NSC_SEGMENTATION,
    nsc_m_rule=NSC_M_RULE,
    nsc_tau=NSC_TAU,
    nsc_gamma=NSC_GAMMA,
    nsc_beta=NSC_BETA,
    nsc_M_min=NSC_MMIN,
    nsc_M_max=NSC_MMAX,
    nsc_l_min=NSC_LMIN,
    assume_standardized=ASSUME_STANDARDIZED,

    # TabPFN-2.5 head
    tabpfn_seed=TABPFN_SEED,
    device=DEVICE,

    return_predictions=True,
    verbose=False,
)


# -----------------------
# Access outputs
# -----------------------
print(results["summary"])

metrics_df = results["metrics_df"]
predictions_df = results["predictions_df"]

print("\nFold-wise metrics")
print(metrics_df)

print("\nPrediction preview")
print(predictions_df.head())

print("\nGO-LR ordering")
print(f"Learned GO-LR order length: {len(results['ordering'])}")
print("First 10 ordered features:")
print(results["ordered_feature_names"][:10])


# -----------------------
# Final 5x5 CV summary
# -----------------------
r2s = metrics_df["r2"].to_numpy()
rmses = metrics_df["rmse"].to_numpy()
maes = metrics_df["mae"].to_numpy()

print("\nFinal 5x5 CV results")
print(f"R2   : {np.mean(r2s):.4f} ± {np.std(r2s, ddof=1):.4f}")
print(f"RMSE : {np.mean(rmses):.4f} ± {np.std(rmses, ddof=1):.4f}")
print(f"MAE  : {np.mean(maes):.4f} ± {np.std(maes, ddof=1):.4f}")

GOTabPFN completed successfully.

Task type: regression
Evaluation: 5x5 repeated CV
Samples: 606
Original numeric features: 6400
GO-LR metric: manhattan
GO-LR KMeans mode: gpu
GO-LR feature subsample: 2000
NSC segmentation: largest_jump
NSC M rule: idf
NSC bypass_if_m_leq: 0
Device: cuda
Elapsed time: 132.81 seconds

Metric summary:
r2: 0.6548 ± 0.0995
rmse: 8.2695 ± 1.4722
mae: 3.6689 ± 0.6156
nsc_tokens: 256.0000 ± 0.0000

Fold-wise metrics
    fold        r2       rmse       mae  nsc_tokens
0      1  0.512414  10.792954  4.808915         256
1      2  0.702385   6.701759  2.874984         256
2      3  0.767160   7.359853  3.626536         256
3      4  0.743784   7.608474  3.411266         256
4      5  0.631153   7.991230  3.200625         256
5      6  0.740289   7.367319  3.588372         256
6      7  0.649674   9.099552  4.290849         256
7      8  0.571329   8.833177  3.676428         256
8      9  0.497356   9.076891  3.413999         256
9     10  0.748292   7.312588  3.

# orlaws10P || Multiclass Classification || Single Wrapper

In [6]:
import numpy as np
import torch

from gotabpfn import run_gotabpfn_csv


# -----------------------
# User settings
# -----------------------
DATA_FILE = "orlraws10P.csv"  # change this to your dataset file name
TARGET_COL = "label"          # change this to your target column
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# -----------------------
# Fixed GO-LR hyperparameters
# -----------------------
GO_METRIC = "cosine"
GO_NUM_CLUSTERS = 5
GO_REFINE_PASSES = 1
GO_DIRECTION_SELECT = False
GO_FEAT_SUBSAMPLE = 3000


# -----------------------
# Fixed NSC-pSP hyperparameters
# -----------------------
NSC_SEGMENTATION = "uniform"
NSC_M_RULE = "default"
NSC_TAU = 0.99
NSC_GAMMA = 2.049512863264476
NSC_BETA = 0.3887505167779042
NSC_MMIN = 32
NSC_MMAX = 384
NSC_LMIN = 12
ASSUME_STANDARDIZED = False

TABPFN_SEED = 42


# -----------------------
# Run GOTabPFN
# -----------------------
results = run_gotabpfn_csv(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    task_type="multiclass",
    non_numeric="drop",

    # 5x5 repeated stratified cross-validation
    cv="5x5",
    seed=SEED,

    # GO-LR settings
    go_metric=GO_METRIC,
    go_num_clusters=GO_NUM_CLUSTERS,
    go_refine=True,
    go_direction_select=GO_DIRECTION_SELECT,
    go_refine_passes=GO_REFINE_PASSES,
    go_feat_subsample=GO_FEAT_SUBSAMPLE,

    # Try GPU KMeans first, then fall back to CPU KMeans if needed
    go_use_cpu_kmeans=False,
    go_fallback_cpu_kmeans=True,

    # NSC-pSP settings
    nsc_segmentation=NSC_SEGMENTATION,
    nsc_m_rule=NSC_M_RULE,
    nsc_tau=NSC_TAU,
    nsc_gamma=NSC_GAMMA,
    nsc_beta=NSC_BETA,
    nsc_M_min=NSC_MMIN,
    nsc_M_max=NSC_MMAX,
    nsc_l_min=NSC_LMIN,
    assume_standardized=ASSUME_STANDARDIZED,

    # TabPFN-2.5 head
    tabpfn_seed=TABPFN_SEED,
    device=DEVICE,

    return_predictions=True,
    verbose=True,
)


# -----------------------
# Access outputs
# -----------------------
print(results["summary"])

metrics_df = results["metrics_df"]
predictions_df = results["predictions_df"]

print("\nFold-wise metrics")
print(metrics_df)

print("\nPrediction preview")
print(predictions_df.head())

print("\nGO-LR ordering")
print(f"Learned GO-LR order length: {len(results['ordering'])}")
print("First 10 ordered features:")
print(results["ordered_feature_names"][:10])


# -----------------------
# Final 5x5 CV summary
# -----------------------
accs = metrics_df["accuracy"].to_numpy()
f1s = metrics_df["macro_f1"].to_numpy()

print("\nFinal 5x5 CV results")
print(f"Accuracy      : {np.mean(accs):.4f} ± {np.std(accs, ddof=1):.4f}")
print(f"Macro-F1      : {np.mean(f1s):.4f} ± {np.std(f1s, ddof=1):.4f}")

if "macro_ovr_auc" in metrics_df.columns:
    aucs = metrics_df["macro_ovr_auc"].dropna().to_numpy()
    if len(aucs) > 0:
        print(f"Macro-OvR-AUC : {np.mean(aucs):.4f} ± {np.std(aucs, ddof=1):.4f}")

X shape: (100, 10304), classes: 10
Using device: cuda
Learned GO-LR order length: 10304
First 10 ordered features:
['feature_339', 'feature_340', 'feature_10195', 'feature_341', 'feature_9633', 'feature_10087', 'feature_116', 'feature_10085', 'feature_10', 'feature_9860']
Fold 01 Z_tr shape: (80, 142)
Fold 01: ACC=1.0000, Macro-F1=1.0000, Macro-OvR-AUC=1.0000
Fold 02 Z_tr shape: (80, 142)
Fold 02: ACC=1.0000, Macro-F1=1.0000, Macro-OvR-AUC=1.0000
Fold 03 Z_tr shape: (80, 142)
Fold 03: ACC=1.0000, Macro-F1=1.0000, Macro-OvR-AUC=1.0000
Fold 04 Z_tr shape: (80, 140)
Fold 04: ACC=1.0000, Macro-F1=1.0000, Macro-OvR-AUC=1.0000
Fold 05 Z_tr shape: (80, 142)
Fold 05: ACC=1.0000, Macro-F1=1.0000, Macro-OvR-AUC=1.0000
Fold 06 Z_tr shape: (80, 142)
Fold 06: ACC=1.0000, Macro-F1=1.0000, Macro-OvR-AUC=1.0000
Fold 07 Z_tr shape: (80, 142)
Fold 07: ACC=1.0000, Macro-F1=1.0000, Macro-OvR-AUC=1.0000
Fold 08 Z_tr shape: (80, 142)
Fold 08: ACC=1.0000, Macro-F1=1.0000, Macro-OvR-AUC=1.0000
Fold 09 Z_tr sh

# Colon || Binary Classification || Single Wrapper

In [2]:
import numpy as np
import torch

from gotabpfn import run_gotabpfn_csv


# -----------------------
# User settings
# -----------------------
DATA_FILE = "coloncancer_encoded.csv"  # change this to your dataset file name
TARGET_COL = "label"                   # change this to your target column
SEED = 42

# Fixed GO-LR hyperparameters
GO_METRIC = "euclidean"
GO_NUM_CLUSTERS = 10
GO_REFINE_PASSES = 3
GO_DIRECTION_SELECT = True
GO_FEAT_SUBSAMPLE = None

# Fixed NSC-pSP hyperparameters
NSC_SEGMENTATION = "equal_mass"
NSC_M_RULE = "idf"
NSC_TAU = 0.99
NSC_GAMMA = 1.7570143129240916
NSC_BETA = 0.2244046472232107
NSC_MMIN = 64
NSC_MMAX = 384
NSC_LMIN = 16
ASSUME_STANDARDIZED = False

TABPFN_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# -----------------------
# Run GOTabPFN
# -----------------------
results = run_gotabpfn_csv(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    task_type="binary",
    non_numeric="drop",

    # 5x5 repeated stratified cross-validation
    cv="5x5",
    seed=SEED,

    # GO-LR settings
    go_metric=GO_METRIC,
    go_num_clusters=GO_NUM_CLUSTERS,
    go_refine=True,
    go_direction_select=GO_DIRECTION_SELECT,
    go_refine_passes=GO_REFINE_PASSES,

    # Try GPU KMeans first, then fall back to CPU KMeans if needed
    go_use_cpu_kmeans=False,
    go_fallback_cpu_kmeans=True,

    # NSC-pSP settings
    nsc_segmentation=NSC_SEGMENTATION,
    nsc_m_rule=NSC_M_RULE,
    nsc_tau=NSC_TAU,
    nsc_gamma=NSC_GAMMA,
    nsc_beta=NSC_BETA,
    nsc_M_min=NSC_MMIN,
    nsc_M_max=NSC_MMAX,
    nsc_l_min=NSC_LMIN,
    assume_standardized=ASSUME_STANDARDIZED,

    # TabPFN-2.5 head
    tabpfn_seed=TABPFN_SEED,
    device=DEVICE,

    return_predictions=True,
    verbose=True,
)


# -----------------------
# Access outputs
# -----------------------
print(results["summary"])

metrics_df = results["metrics_df"]
predictions_df = results["predictions_df"]

print("\nFold-wise metrics")
print(metrics_df)

print("\nPrediction preview")
print(predictions_df.head())

print("\nGO-LR ordering")
print(f"Learned GO-LR order length: {len(results['ordering'])}")
print("First 10 ordered features:")
print(results["ordered_feature_names"][:10])


# -----------------------
# Final 5x5 CV summary
# -----------------------
accs = metrics_df["accuracy"].to_numpy()
f1s = metrics_df["macro_f1"].to_numpy()
aucs = metrics_df["auc"].dropna().to_numpy()

print("\nFinal 5x5 CV results")
print(f"Accuracy : {np.mean(accs):.4f} ± {np.std(accs, ddof=1):.4f}")
print(f"Macro-F1 : {np.mean(f1s):.4f} ± {np.std(f1s, ddof=1):.4f}")
print(f"AUC      : {np.mean(aucs):.4f} ± {np.std(aucs, ddof=1):.4f}")

X shape: (62, 2000), classes: 2
Using device: cuda
Learned GO-LR order length: 2000
First 10 ordered features:
['138', '1244', '1912', '1242', '381', '692', '25', '248', '697', '211']
Fold 01 Z_tr shape: (49, 64)
Fold 01: ACC=1.0000, F1=1.0000, AUC=1.0000
Fold 02 Z_tr shape: (49, 64)
Fold 02: ACC=0.8462, F1=0.8375, AUC=0.9000
Fold 03 Z_tr shape: (50, 64)
Fold 03: ACC=0.9167, F1=0.9111, AUC=1.0000
Fold 04 Z_tr shape: (50, 64)
Fold 04: ACC=0.8333, F1=0.8125, AUC=0.8438
Fold 05 Z_tr shape: (50, 64)
Fold 05: ACC=1.0000, F1=1.0000, AUC=1.0000
Fold 06 Z_tr shape: (49, 64)
Fold 06: ACC=1.0000, F1=1.0000, AUC=1.0000
Fold 07 Z_tr shape: (49, 64)
Fold 07: ACC=0.6154, F1=0.5752, AUC=0.6750
Fold 08 Z_tr shape: (50, 64)
Fold 08: ACC=1.0000, F1=1.0000, AUC=1.0000
Fold 09 Z_tr shape: (50, 64)
Fold 09: ACC=0.8333, F1=0.8125, AUC=0.9375
Fold 10 Z_tr shape: (50, 64)
Fold 10: ACC=0.9167, F1=0.8992, AUC=1.0000
Fold 11 Z_tr shape: (49, 64)
Fold 11: ACC=0.8462, F1=0.8375, AUC=0.9000
Fold 12 Z_tr shape: (49,

# Colon || Binary Classification || Manual Approach

In [3]:
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from gotabpfn import GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config


# -----------------------
# User settings
# -----------------------
DATA_FILE = "coloncancer_encoded.csv"  # change your dataset file name
TARGET_COL = "label"                   # change your dataset target column
SEED = 42

# Fixed GOTabPFN hyperparameters
GO_METRIC = "euclidean"
GO_NUM_CLUSTERS = 10
GO_REFINE_PASSES = 3
GO_DIRECTION_SELECT = True

NSC_SEGMENTATION = "equal_mass"
NSC_M_RULE = "idf"
NSC_TAU = 0.99
NSC_GAMMA = 1.7570143129240916
NSC_BETA = 0.2244046472232107
NSC_MMIN = 64
NSC_MMAX = 384
NSC_LMIN = 16
ASSUME_STANDARDIZED = False

TABPFN_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# -----------------------
# Utility
# -----------------------
def compute_deltas_adjacent_corr(X_tr, Pi_star, eps=1e-12):
    """
    Compute adjacent transition scores along the GO-LR order:
        delta_t = 1 - |corr(feature_t, feature_{t+1})|.

    Required for transition-aware NSC segmentation rules:
        - equal_mass
        - largest_jump
    """
    X_t = torch.from_numpy(X_tr).float()
    perm = torch.tensor(Pi_star, dtype=torch.long)

    Xp = X_t[:, perm]
    Xc = Xp - Xp.mean(dim=0, keepdim=True)
    std = Xc.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
    Z = Xc / std

    corr_adj = (Z[:, :-1] * Z[:, 1:]).mean(dim=0)
    deltas = 1.0 - corr_adj.abs()

    return deltas.cpu()


# -----------------------
# Load and preprocess data
# -----------------------
df = pd.read_csv(DATA_FILE)

if TARGET_COL not in df.columns:
    raise ValueError(f"TARGET_COL='{TARGET_COL}' not found in the CSV file.")

y_raw = df[TARGET_COL].astype(str).fillna("missing_target")
X_df = df.drop(columns=[TARGET_COL])

# Keep numeric features only
X_df = X_df.select_dtypes(include=[np.number])
X_df = X_df.apply(pd.to_numeric, errors="coerce")
X_df = X_df.fillna(X_df.median(numeric_only=True)).fillna(0.0)

if X_df.shape[1] == 0:
    raise ValueError("No numeric feature columns found after preprocessing.")

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y_raw).astype(np.int64)

num_classes = len(le.classes_)
if num_classes != 2:
    raise ValueError(
        f"This example expects binary classification, but found {num_classes} classes."
    )

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X_df.values).astype(np.float32)

print(f"X shape: {X.shape}")
print(f"Classes: {list(le.classes_)}")
print(f"Using device: {DEVICE}")


# -----------------------
# Learn GO-LR feature ordering once
# -----------------------
go = GraphFeatureOrdering(
    num_clusters=GO_NUM_CLUSTERS,
    metric=GO_METRIC,
    refine=True,
    direction_select=GO_DIRECTION_SELECT,
    refine_passes=GO_REFINE_PASSES,
)

try:
    Pi_star, _, _, _ = go.fit(
        X,
        seed=SEED,
        deterministic=True,
        use_cpu_kmeans=False,
    )
except Exception:
    Pi_star, _, _, _ = go.fit(
        X,
        seed=SEED,
        deterministic=True,
        use_cpu_kmeans=True,
    )

Pi_star = list(map(int, Pi_star))

print(f"Learned GO-LR order length: {len(Pi_star)}")


# -----------------------
# 5x5 cross-validation
# -----------------------
rkf = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=SEED,
)

head_cfg = TabPFN25Config(
    task_type="binary",
    num_classes=2,
    device=DEVICE,
    random_state=TABPFN_SEED,
)

accs, f1s, aucs = [], [], []

for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X, y), start=1):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    nsc = pidf_segpca(
        segmentation=NSC_SEGMENTATION,
        l_min=NSC_LMIN,
        m_rule=NSC_M_RULE,
        gamma=NSC_GAMMA,
        beta=NSC_BETA,
        tau=NSC_TAU,
        M_min=NSC_MMIN,
        M_max=NSC_MMAX,
        assume_standardized=ASSUME_STANDARDIZED,
        device=DEVICE,
    )

    X_tr_t = torch.from_numpy(X_tr)

    # equal_mass and largest_jump require transition scores.
    deltas = None
    if NSC_SEGMENTATION in {"equal_mass", "largest_jump"}:
        deltas = compute_deltas_adjacent_corr(X_tr, Pi_star)

    nsc.configure(
        Pi_star=Pi_star,
        X_train=X_tr_t,
        tau=NSC_TAU,
        deltas=deltas,
    )

    Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
    Z_va = nsc.compress(torch.from_numpy(X_va), mode="flatten").cpu().numpy()

    head = TabPFN25Head(head_cfg)
    head.fit(Z_tr, y_tr)

    P = head.predict_proba(Z_va)
    pred = np.argmax(P, axis=1)

    acc = accuracy_score(y_va, pred)
    f1 = f1_score(y_va, pred, average="macro")
    auc = roc_auc_score(y_va, P[:, 1])

    accs.append(acc)
    f1s.append(f1)
    aucs.append(auc)

    print(f"Fold {fold_id:02d}: ACC={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}")


print("\nFinal 5x5 CV results")
print(f"Accuracy : {np.mean(accs):.4f} ± {np.std(accs, ddof=1):.4f}")
print(f"Macro-F1 : {np.mean(f1s):.4f} ± {np.std(f1s, ddof=1):.4f}")
print(f"AUC      : {np.mean(aucs):.4f} ± {np.std(aucs, ddof=1):.4f}")

X shape: (62, 2000)
Classes: ['0', '1']
Using device: cuda
Learned GO-LR order length: 2000
Fold 01: ACC=1.0000, F1=1.0000, AUC=1.0000
Fold 02: ACC=0.8462, F1=0.8375, AUC=0.9000
Fold 03: ACC=0.9167, F1=0.9111, AUC=1.0000
Fold 04: ACC=0.8333, F1=0.8125, AUC=0.8438
Fold 05: ACC=1.0000, F1=1.0000, AUC=1.0000
Fold 06: ACC=1.0000, F1=1.0000, AUC=1.0000
Fold 07: ACC=0.6154, F1=0.5752, AUC=0.6750
Fold 08: ACC=1.0000, F1=1.0000, AUC=1.0000
Fold 09: ACC=0.8333, F1=0.8125, AUC=0.9375
Fold 10: ACC=0.9167, F1=0.8992, AUC=1.0000
Fold 11: ACC=0.8462, F1=0.8375, AUC=0.9000
Fold 12: ACC=0.8462, F1=0.8375, AUC=0.9250
Fold 13: ACC=0.9167, F1=0.9111, AUC=0.9688
Fold 14: ACC=0.8333, F1=0.8286, AUC=0.8125
Fold 15: ACC=0.9167, F1=0.8992, AUC=0.8438
Fold 16: ACC=0.8462, F1=0.8375, AUC=0.9000
Fold 17: ACC=1.0000, F1=1.0000, AUC=1.0000
Fold 18: ACC=1.0000, F1=1.0000, AUC=1.0000
Fold 19: ACC=0.9167, F1=0.9111, AUC=1.0000
Fold 20: ACC=0.8333, F1=0.8125, AUC=0.7500
Fold 21: ACC=0.7692, F1=0.7451, AUC=0.8750
Fold 

In [4]:
import gotabpfn
import tabpfn
import inspect

print("gotabpfn version:", gotabpfn.__version__)
print("gotabpfn path:", gotabpfn.__file__)
print("run_gotabpfn_csv file:", inspect.getsourcefile(gotabpfn.run_gotabpfn_csv))

print("tabpfn version:", getattr(tabpfn, "__version__", "NO __version__"))
print("tabpfn path:", tabpfn.__file__)

gotabpfn version: 0.1.11
gotabpfn path: /usr/local/lib/python3.12/dist-packages/gotabpfn/__init__.py
run_gotabpfn_csv file: /usr/local/lib/python3.12/dist-packages/gotabpfn/gotabpfn.py
tabpfn version: 6.3.1
tabpfn path: /usr/local/lib/python3.12/dist-packages/tabpfn/__init__.py
